In [2]:
%pip install pandas pyarrow scikit-learn mlxtend

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow.parquet as pq

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import GroupShuffleSplit
from mlxtend.frequent_patterns import fpgrowth, association_rules

DATA_PATH = Path(
    r"E:\Dataminingproject\Data\reddit_climate_stance_scored.parquet"
)

OUTPUT_DIR = Path(r"E:\Dataminingproject\ARM_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMUNITIES = [
    "climatechange",
    "climate",
    "energy",
    "Futurology",
    "conspiracy",
    "changemyview",
]

START = pd.Timestamp("2023-11-19")
END = pd.Timestamp("2026-09-01")  # 不包含这一天

SEED = 42

In [4]:
columns = [
    "comment_id",
    "post_id",
    "subreddit",
    "created_time",
    "clean_text",
    "comment_word_count",
    "stance_score",
]

rng = np.random.default_rng(SEED)
pools = {}

file = pq.ParquetFile(DATA_PATH)

for batch in file.iter_batches(batch_size=50_000, columns=columns):
    part = batch.to_pandas()

    keep = (
        part["subreddit"].isin(COMMUNITIES)
        & part["created_time"].ge(START)
        & part["created_time"].lt(END)
        & part["comment_word_count"].ge(5)
        & part["clean_text"].notna()
        & part["stance_score"].between(0, 1)
    )

    part = part.loc[keep].copy()
    part = part[part["clean_text"].str.strip().ne("")]

    part["month"] = part["created_time"].dt.strftime("%Y-%m")
    part["_random"] = rng.random(len(part))

    for key, group in part.groupby(["subreddit", "month"]):
        if key in pools:
            group = pd.concat([pools[key], group], ignore_index=True)

        pools[key] = group.nsmallest(100, "_random")

df = pd.concat(pools.values(), ignore_index=True)

df = (
    df.sort_values("_random")
    .groupby("post_id", sort=False)
    .head(10)
    .reset_index(drop=True)
)

assert df["comment_id"].is_unique
assert df.groupby("post_id").size().max() <= 10

print("样本数量：", len(df))
display(df["subreddit"].value_counts())
display(df[["clean_text", "stance_score"]].head())

样本数量： 20136


subreddit
conspiracy       3400
climatechange    3392
climate          3390
changemyview     3388
Futurology       3319
energy           3247
Name: count, dtype: int64[pyarrow]

,clean_text,stance_score
0,Read the 1996 IPCC foundational documents on w...,0.988131
1,"you don't think you have a religion, but you d...",0.532182
2,Well we don't need every thread to be the same...,0.371593
3,Agreed. I’ve genuinely found God at this point...,0.574633
4,"I will absolutely be checking those out, thank...",0.910147


In [5]:
df["stance_group"] = np.select(
    [
        df["stance_score"].le(0.3),
        df["stance_score"].ge(0.7),
    ],
    ["low", "high"],
    default="middle",
)

display(df["stance_group"].value_counts())

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED,
)

train_index, test_index = next(
    splitter.split(df, groups=df["post_id"])
)

train = df.iloc[train_index].reset_index(drop=True)
test = df.iloc[test_index].reset_index(drop=True)

assert set(train["post_id"]).isdisjoint(set(test["post_id"]))

print("用于发现规则：", len(train))
print("用于验证规则：", len(test))

stance_group
middle    10650
high       6915
low        2571
Name: count, dtype: int64

用于发现规则： 16063
用于验证规则： 4073


In [6]:
vectorizer = CountVectorizer(
    stop_words="english",
    ngram_range=(1, 2),
    min_df=0.01,
    max_df=0.5,
    max_features=200,
    binary=True,
)

train_words = vectorizer.fit_transform(train["clean_text"])
test_words = vectorizer.transform(test["clean_text"])

word_columns = [
    "word:" + word
    for word in vectorizer.get_feature_names_out()
]

train_basket = pd.DataFrame(
    train_words.toarray().astype(bool),
    columns=word_columns,
)

test_basket = pd.DataFrame(
    test_words.toarray().astype(bool),
    columns=word_columns,
)

for group in ["low", "middle", "high"]:
    train_basket["stance:" + group] = train["stance_group"].eq(group)
    test_basket["stance:" + group] = test["stance_group"].eq(group)

print("提取的词和词组：")
print(vectorizer.get_feature_names_out())

display(train_basket.head())

提取的词和词组：
['10' '100' '20' 'able' 'actually' 'ago' 'agree' 'ai' 'aren' 'away' 'bad'
 'based' 'believe' 'best' 'better' 'big' 'build' 'carbon' 'care' 'cause'
 'change' 'china' 'climate' 'climate change' 'co2' 'coal' 'come'
 'companies' 'control' 'cost' 'countries' 'country' 'course' 'current'
 'data' 'day' 'decades' 'did' 'didn' 'different' 'does' 'doesn' 'doing'
 'don' 'don think' 'earth' 'electricity' 'emissions' 'end' 'energy'
 'evidence' 'exactly' 'example' 'fact' 'far' 'feel' 'food' 'fossil'
 'fossil fuels' 'free' 'fuel' 'fuels' 'future' 'gas' 'gets' 'getting'
 'global' 'god' 'going' 'good' 'got' 'government' 'great' 'happen' 'hard'
 'having' 'heat' 'help' 'high' 'higher' 'history' 'huge' 'human' 'humans'
 'idea' 'industry' 'instead' 'isn' 'issue' 'just' 'kind' 'know' 'large'
 'let' 'level' 'life' 'like' 'likely' 'literally' 'little' 'live' 'living'
 'll' 'lol' 'long' 'look' 'lot' 'low' 'make' 'makes' 'making' 'matter'
 'maybe' 'mean' 'means' 'money' 'natural' 'need' 'needs' 'new' '

,word:10,word:100,word:20,word:able,word:actually,word:ago,word:agree,word:ai,word:aren,word:away,...,word:worse,word:wouldn,word:wrong,word:yeah,word:year,word:years,word:yes,stance:low,stance:middle,stance:high
0,False,False,False,False,False,False,False,False,False,True,...,False,False,False,False,False,False,False,False,False,True
1,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
2,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,True,False
3,False,False,False,False,False,False,False,True,False,False,...,False,False,False,False,False,False,False,False,True,False
4,False,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,True


In [7]:
frequent_items = fpgrowth(
    train_basket,
    min_support=0.005,
    use_colnames=True,
    max_len=2,
)

if not frequent_items["itemsets"].map(len).ge(2).any():
    raise ValueError("没有足够频繁的二项组合，请先检查词表和样本。")

rules = association_rules(
    frequent_items,
    metric="confidence",
    min_threshold=0.6,
)

# 只保留：词或词组 → 高分／低分
wanted = (
    rules["antecedents"].apply(
        lambda items:
        len(items) == 1
        and all(item.startswith("word:") for item in items)
    )
    & rules["consequents"].apply(
        lambda items:
        len(items) == 1
        and next(iter(items)) in {"stance:high", "stance:low"}
    )
    & rules["lift"].ge(1.2)
)

rules = rules.loc[wanted].copy()

rules["word"] = rules["antecedents"].apply(
    lambda items: next(iter(items)).removeprefix("word:")
)
rules["target"] = rules["consequents"].apply(
    lambda items: next(iter(items)).removeprefix("stance:")
)

rules = rules.sort_values(
    ["lift", "support"],
    ascending=False,
).reset_index(drop=True)

display(
    rules[["word", "target", "support", "confidence", "lift"]].head(20)
)

,word,target,support,confidence,lift
0,solar,high,0.031252,0.817590,2.381745
1,renewables,high,0.012513,0.794466,2.314384
2,fuels,high,0.014692,0.784053,2.284049
3,fossil fuels,high,0.013074,0.780669,2.274191
4,fossil,high,0.022723,0.762004,2.219817
5,wind,high,0.017058,0.752747,2.192851
6,electricity,high,0.012202,0.736842,2.146517
7,coal,high,0.012700,0.723404,2.107371
8,fuel,high,0.016871,0.696658,2.029456
9,emissions,high,0.018490,0.676538,1.970842


In [8]:
checks = []

for _, rule in rules.iterrows():
    word_present = test_basket["word:" + rule["word"]]
    target_present = test_basket["stance:" + rule["target"]]

    word_count = int(word_present.sum())
    joint_count = int((word_present & target_present).sum())
    baseline = float(target_present.mean())

    confidence = (
        joint_count / word_count if word_count else np.nan
    )
    lift = (
        confidence / baseline if baseline > 0 else np.nan
    )

    checks.append({
        "word": rule["word"],
        "target": rule["target"],
        "train_support": rule["support"],
        "train_confidence": rule["confidence"],
        "train_lift": rule["lift"],
        "test_word_count": word_count,
        "test_joint_count": joint_count,
        "test_confidence": confidence,
        "test_lift": lift,
    })

checked = pd.DataFrame(checks, columns=[
    "word", "target",
    "train_support", "train_confidence", "train_lift",
    "test_word_count", "test_joint_count",
    "test_confidence", "test_lift",
])

display(checked.head(20))

,word,target,train_support,train_confidence,train_lift,test_word_count,test_joint_count,test_confidence,test_lift
0,solar,high,0.031252,0.817590,2.381745,171,139,0.812865,2.363170
1,renewables,high,0.012513,0.794466,2.314384,50,46,0.920000,2.674632
2,fuels,high,0.014692,0.784053,2.284049,81,64,0.790123,2.297054
3,fossil fuels,high,0.013074,0.780669,2.274191,73,58,0.794521,2.309837
4,fossil,high,0.022723,0.762004,2.219817,145,109,0.751724,2.185419
5,wind,high,0.017058,0.752747,2.192851,77,59,0.766234,2.227602
6,electricity,high,0.012202,0.736842,2.146517,64,50,0.781250,2.271257
7,coal,high,0.012700,0.723404,2.107371,72,52,0.722222,2.099651
8,fuel,high,0.016871,0.696658,2.029456,114,80,0.701754,2.040147
9,emissions,high,0.018490,0.676538,1.970842,118,76,0.644068,1.872440


In [9]:
if not checked.empty:
    first = checked.iloc[0]

    word_mask = test_basket["word:" + first["word"]]
    target_mask = test_basket["stance:" + first["target"]]

    examples = test.loc[
        word_mask & target_mask,
        ["subreddit", "clean_text", "stance_score"],
    ]

    print("规则：", first["word"], "→", first["target"])

    pd.set_option("display.max_colwidth", 500)
    display(examples.head(10))
else:
    print("当前设置下，没有符合条件的规则。")

checked.to_csv(
    OUTPUT_DIR / "word_stance_rules.csv",
    index=False,
    encoding="utf-8-sig",
)

print("结果保存到：", OUTPUT_DIR / "word_stance_rules.csv")

规则： solar → high


,subreddit,clean_text,stance_score
127,climatechange,"We've been investing in personal resilience because we believe that prices will only continue to go up, especially with supply chain disruptions and more. This might not be true for solar/battery though, those prices seem to be dropping lately. If index funds were no longer a concern, I believe that might be TEOTWAWKI, or the end of the world as we know it. I hope to never see that (unless it's on the way to a Star Trek utopia) but we're having a lot of fun working on our off grid property. ...",0.978814
171,energy,"Basically geothermal storage: you take a large amount of unused land, bury a lot of pipes deep underground, then during summer heat up water with the leftover solar and force it down the pipes. Due to how insulating soil is you're left with soil that is still hot during winter, at which point you run water through it again to heat your home. It's already used in the field to heat homes all year long on 97% solar power.",0.893774
289,energy,Big opportunity for us in the Summer to sell excess solar power to Canada and Canada sells us excess hydro for our heating in Winter,0.982078
323,climatechange,Solar is kind of the best energy source for hot places that use HVAC to keep cool. Solar is generated the most on days and time f the day where it is needed the most.,0.984000
384,Futurology,There as never been a more American response to renewable power... American plan to replace Petroleum: A. Solar: No! B. Wind: No! C. Geothermal: No! D. Tidal. No! E. Cow shit... Hell yeah!,0.819587
400,energy,"Most wind and solar are in rural conservative districts, and the money stays local. The taxes from the development go to schools, libraries, fire departments, and other taxing bodies; or to keep residents' property taxes low. Payments to landowners are significant, and the jobs are mostly local too. Those folks spend locally. All this investment and economic development coming into places full of republican constituents who will see huge benefits, and they're not only waging a war against wi...",0.924790
416,Futurology,Battery technology has already made solar + battery cheaper than new coal. It’s over for fossil fuel energy.,0.995180
426,energy,What does solar and wind energy charge?,0.933971
433,energy,"Hi all, I hope you'll read my latest column for the L.A. Times and let me know what you think. Here's how it starts: >*“I want you to stop changing your lives, OK? We’ve been conditioned to think that change is good and exciting, but what if it’s not? What if it’s actually bad and very, very dangerous?”* >*So says the Janitor, played by Neil Flynn, in one of the final episodes of my favorite TV show, “Scrubs.” It’s a silly-but-serious line that left a deep impression on me when I was a 16-ye...",0.991067
453,climate,"China and India are the world’s only hope at this point. If china can cut down fast enough while India keeps skipping over oil for solar, wind and hydro, then this world might have a chance to avoid catastrophe.",0.981806


结果保存到： E:\Dataminingproject\ARM_results\word_stance_rules.csv


In [10]:
word = "solar"
has_word = test_basket["word:" + word]

print("包含该词、模型评为 high：")
display(
    test.loc[
        has_word & test["stance_group"].eq("high"),
        ["subreddit", "clean_text", "stance_score"],
    ].head(10)
)

print("包含该词、但模型没有评为 high：")
display(
    test.loc[
        has_word & test["stance_group"].ne("high"),
        ["subreddit", "clean_text", "stance_score"],
    ].head(10)
)

包含该词、模型评为 high：


,subreddit,clean_text,stance_score
127,climatechange,"We've been investing in personal resilience because we believe that prices will only continue to go up, especially with supply chain disruptions and more. This might not be true for solar/battery though, those prices seem to be dropping lately. If index funds were no longer a concern, I believe that might be TEOTWAWKI, or the end of the world as we know it. I hope to never see that (unless it's on the way to a Star Trek utopia) but we're having a lot of fun working on our off grid property. ...",0.978814
171,energy,"Basically geothermal storage: you take a large amount of unused land, bury a lot of pipes deep underground, then during summer heat up water with the leftover solar and force it down the pipes. Due to how insulating soil is you're left with soil that is still hot during winter, at which point you run water through it again to heat your home. It's already used in the field to heat homes all year long on 97% solar power.",0.893774
289,energy,Big opportunity for us in the Summer to sell excess solar power to Canada and Canada sells us excess hydro for our heating in Winter,0.982078
323,climatechange,Solar is kind of the best energy source for hot places that use HVAC to keep cool. Solar is generated the most on days and time f the day where it is needed the most.,0.984000
384,Futurology,There as never been a more American response to renewable power... American plan to replace Petroleum: A. Solar: No! B. Wind: No! C. Geothermal: No! D. Tidal. No! E. Cow shit... Hell yeah!,0.819587
400,energy,"Most wind and solar are in rural conservative districts, and the money stays local. The taxes from the development go to schools, libraries, fire departments, and other taxing bodies; or to keep residents' property taxes low. Payments to landowners are significant, and the jobs are mostly local too. Those folks spend locally. All this investment and economic development coming into places full of republican constituents who will see huge benefits, and they're not only waging a war against wi...",0.924790
416,Futurology,Battery technology has already made solar + battery cheaper than new coal. It’s over for fossil fuel energy.,0.995180
426,energy,What does solar and wind energy charge?,0.933971
433,energy,"Hi all, I hope you'll read my latest column for the L.A. Times and let me know what you think. Here's how it starts: >*“I want you to stop changing your lives, OK? We’ve been conditioned to think that change is good and exciting, but what if it’s not? What if it’s actually bad and very, very dangerous?”* >*So says the Janitor, played by Neil Flynn, in one of the final episodes of my favorite TV show, “Scrubs.” It’s a silly-but-serious line that left a deep impression on me when I was a 16-ye...",0.991067
453,climate,"China and India are the world’s only hope at this point. If china can cut down fast enough while India keeps skipping over oil for solar, wind and hydro, then this world might have a chance to avoid catastrophe.",0.981806


包含该词、但模型没有评为 high：


,subreddit,clean_text,stance_score
346,climate,Nooooooo solar free energy is stupid and and a communist /s,0.048903
419,energy,"They will use this as an excuse to pull funding from solar for 'national security reasons'... leaving our actual energy security up to the whims of OPEC, which doesn't seem very secure to me.",0.561696
650,energy,The generators are angry because of solar and wind they have to lower output more and make less. The horror.,0.610918
839,conspiracy,Was it a solar or a lunar?,0.527803
1165,climatechange,"Two key questions to ask your dad: - Since fires happened in the past due to natural causes, can we therefore claim that fires can only ever happen due to natural causes? - How do we know that the Earth's climate changed in the past? A: Well it was only through science that showed us that the climate changed in the past, and science has now shown us that the recent rapid warming of the Earth is due mankind increasing the amount of CO2 and methane in the atmosphere &nbsp; Here's a video clip ...",0.151655
1237,Futurology,Then we wonder why we were so excited about fusion when wind solar and batteries are working so well. Maybe space flight is where it gets used,0.491212
1302,Futurology,Solar and wind are okay if you don't mind fully deindustrializing your country and not participating in manufacturing. That is why China is building nuclear and the US isn't. That is also why shutting those reactors down was so devastating for Germany. It's simple to understand why nuclear is necessary if you want to produce things.,0.613237
1539,Futurology,"There is a huge leap being made here from “net energy positive” to “cheap, plentiful energy”. Experimental fusion reactor designs right now are incredibly expensive and complex. On top of that, fusion reactions are notoriously difficult both to sustain for any amount of time without damaging the reactor. For instance, fusion reactions tend to produce a lot of neutrons that are energetic enough to cause microscopic (thermal) wear and tear in minutes of operation. This neutron bombardment cann...",0.645567
1554,Futurology,All those blue water navy ships they building (and their general military build up) ain't gonna run on solar power.,0.602880
1561,Futurology,It’s on a kWh basis. The report linked provides levelised cost of energy for solar and storage as well as storage by itself. It’s by lazards who can’t be accused of bias.,0.370681


In [11]:
word = "fossil fuels"
has_word = test_basket["word:" + word]

print("包含该词、模型评为 high：")
display(
    test.loc[
        has_word & test["stance_group"].eq("high"),
        ["subreddit", "clean_text", "stance_score"],
    ].head(10)
)

print("包含该词、但模型没有评为 high：")
display(
    test.loc[
        has_word & test["stance_group"].ne("high"),
        ["subreddit", "clean_text", "stance_score"],
    ].head(10)
)

包含该词、模型评为 high：


,subreddit,clean_text,stance_score
2,conspiracy,"ChemAg and fossil fuels subsidized our caloric needs. In getting rid of so much of the animal/manual labor, and changing the plants to focus solely on yield at the expense of it's hardiness, we have become addicted and are stuck in a death spiral. We use the equivalent of 10 oil ""calories"" for every edible calorie we get out of the ground. We can't transition to renewables because the energy return on energy investment is at best 2:1, but when you add in battery backups, you go below 1:1. Un...",0.897111
277,climate,"One way to do it: 1. Work out where the misunderstanding is; 2. Apologise for your contribution to the misunderstanding; 3. set the record straight To use your example. 1. The misunderstanding has come about because ""AC burns fossil fuels"" could be interpreted to mean that you believe that combustion of fossil fuels happens within the AC unit. You might think that that's ridiculous, and that no one would ever think that. Well, it's easy to underestimate the stupidity of the stupid, and there...",0.958654
353,climatechange,Fossil fuels are a death sentence.,0.964069
372,climate,"Fracking is not a monolith of climate problems, it's a small piece of the problem and dying on that hill right now risks losing the wider war. We can still make progress and create an unfavorable market for fossil fuels with Harris as president. The same cannot be said for Trump who will take us backwards.",0.994008
433,energy,"Hi all, I hope you'll read my latest column for the L.A. Times and let me know what you think. Here's how it starts: >*“I want you to stop changing your lives, OK? We’ve been conditioned to think that change is good and exciting, but what if it’s not? What if it’s actually bad and very, very dangerous?”* >*So says the Janitor, played by Neil Flynn, in one of the final episodes of my favorite TV show, “Scrubs.” It’s a silly-but-serious line that left a deep impression on me when I was a 16-ye...",0.991067
439,climate,"It is a solid solution in all of its forms at this point. Even if we did stop burning fossil fuels tomorrow, we would still have billions of tons of carbon in the atmosphere that shouldn't be there. Nature can't take care of it in a reasonable time either. It would still take over 100,000 years at a minimum and is also not likely more to like 300,000 years, and we're already seeing biodiversity collapse now. If we don't invest and get this working, the earth will become near uninhabitable.",0.999304
487,climate,"Apart from we could greatly reduce the amount of energy needed to produce our food if we put our minds to it, there's no reason why that energy couldn't come from other sources than burning fossil carbon. Also climate change will slow ""economic growth"" so infact divesting from fossil fuels is becoming the primary concern of many governments.",0.990732
563,climate,Global demand grew by 2%. Renewable supply grew by 8%. Given the math on that at some point renewables will start replacing fossil fuels.,0.971725
625,energy,"\>that doesn’t need rebuilding. and there in lies the problem. We can't continue with fossil fuels, so something has to change, and dragging our heels just gives us the worst of both worlds.",0.997513
627,climatechange,"For fires and decomposition, I would argue that while simple, it holds true. Changing the amount of forest available (in land usage) does tip the balance of carbon in the cycle, but ultimately, it's all water sloshing around in the same lazy river. Pushing the water in and out of different segments neither counts as adding or getting rid of water. Meanwhile, fossil fuels are like a garden hose, adding water to the overall system. In Canada for instance, the Arctic is greening rapidly, and fo...",0.769254


包含该词、但模型没有评为 high：


,subreddit,clean_text,stance_score
92,climate,"I'm not a Canadian, but I've seen (and posted) enough stories about Danielle Smith to wonder how this can be considered a betrayal. She's long been in favor of anything that increases the usage of fossil fuels, so this is completely in character for her.",0.132838
437,energy,Fossil president likes fossil fuels.,0.643777
451,Futurology,China isn't investing in renewable energy for ideological reasons. Rather because China has to import the majority of its oil from the Middle East (which as we can see at the moment can be quite unreliable source). China's main goal however is economic growth and fossil fuels are relatively cheap,0.164248
609,climatechange,"If the data differs from your expectations, you conclude that the data must be wrong? CO2 can surely outgas very quickly from water. Ever drop a soda jug then open the top? Perhaps it can be absorbed back into the ocean quickly, both by chemical means and uptake by plants (algae, phytoplankton). Cold water can absorb significant CO2, then sink to the bottom. An example being where ice-melt flows into the ocean. It certainly can be significant since in normal years 10x more CO2 exchanges with...",0.104660
796,climatechange,"So going back to your original claim: > It's a thermodynamic dead end. We get energy from burning fossil fuels. Reversing that process (pulling CO2 out of the air and locking it underground) demands energy. In fact, it takes more energy to reverse the process than we get from burning the fuels in the first place. Does separating co2 from the atmosphere and reinjecting it into wells take more energy than burning the fuel produced in the first place? Or are you going to continue tying yourself...",0.133528
896,climatechange,"Honestly it doesn’t make sense. The effort needed to build such a thing in the first place and all the fossil fuels consumed in construction, manufacturing, etc would be very counterproductive. Massive amounts of water and energy would be needed to maintain that pyramid. Glass cleaners would need to clean it regularly. Maintenance in general will be very high. If there’s even just ONE person driving there everyday for work, it would probably be enough to to offset the captured carbon. There ...",0.638835
1579,energy,I’m a geochemist. Continued use of fossil fuels has led to the initiation of earth’s 6th mass extinction. Human use of fire brought on the dawn of civilization. Continuation will lead to the end of civilization!,0.685213
1610,climatechange,The bigger problem is no convincing long term solution to replace fossil fuels,0.422147
2150,climatechange,"That’s a great question, and it’s smart of you to want to check what you’re being told instead of just accepting it. Here’s a clear way to think about it: 1. Yes, it is true that the Earth’s climate has always been changing, but scientists already take that into account. Technically, it is true that Earth has gone through ice ages and warm periods long before humans existed. Those changes happened because of natural factors in God's design, like: Shifts in Earth’s orbit and tilt (Milankovitc...",0.296654
2160,energy,"You're right. $7 trillion is for all fossil fuels, not just oil.",0.583441


In [13]:
from pathlib import Path
import pandas as pd

# 确认两张表的行一一对应
assert len(test) == len(test_basket)

selected = []

for word in ["solar", "fossil fuels"]:
    has_word = test_basket["word:" + word].to_numpy()

    matching = test.loc[
        has_word,
        ["comment_id", "subreddit", "clean_text",
         "stance_score", "stance_group"]
    ].copy()

    matching["word"] = word

    # 每个词：随机抽最多 10 条高分、10 条非高分
    for category, group in [
        ("high", matching[matching["stance_group"] == "high"]),
        ("not_high", matching[matching["stance_group"] != "high"]),
    ]:
        picked = group.sample(
            n=min(10, len(group)),
            random_state=42,
        ).copy()

        picked["sample_category"] = category
        selected.append(picked)

review = pd.concat(selected, ignore_index=True)

# 留给你人工填写
review["human_label"] = ""
review["reason_or_note"] = ""

review = review[
    [
        "word",
        "sample_category",
        "subreddit",
        "stance_score",
        "stance_group",
        "clean_text",
        "human_label",
        "reason_or_note",
        "comment_id",
    ]
]

output = Path(
    r"E:\Dataminingproject\ARM_results\manual_review.csv"
)
output.parent.mkdir(parents=True, exist_ok=True)

review.to_csv(output, index=False, encoding="utf-8-sig")

print("抽取评论数：", len(review))
print("已保存到：", output)
display(review[["word", "sample_category", "subreddit",
                "stance_score", "clean_text"]])

抽取评论数： 40
已保存到： E:\Dataminingproject\ARM_results\manual_review.csv


,word,sample_category,subreddit,stance_score,clean_text
0,solar,high,energy,0.939969,"360% growth gets 2000TW by 2033 for electrolysis. 197tw of solar at 85% growth. Those numbers are just capacity additions per year. 2040 total energy demand of 205qwh can be met just by 2033 solar additions, at below 4 sun hours/day. Growth rates are usually technically sustainable, because everyone who helped expand a category last year has 1 more year of experience to be promoted to keep the expansion rate."
1,solar,high,Futurology,0.832781,Global electricity demand is growing by single digits annually. Solar is grew 58% last year.
2,solar,high,energy,0.999274,"I referenced that post to point out the difficulties in ramping up of solar production. That is not misinformation, that’s reality. I’m all for solar, don’t presume I’m not. Remember the main reason we are going to solar, to reduce emissions drastically so we don’t see the worst effects of c02 driven climate change. We are already seeing those effects though and it’s only going to get worse. So back to my main point, we need to transition quickly and that is not easy. Even then fossil fuels ..."
3,solar,high,Futurology,0.861887,"What's the cost? Energy storage (and the excess capacity to feed into it) has a cost. Ditto nuclear has a cost. One likely is cheaper than the other. Some of that is going to be situationally dependent. Hydro, wind and solar will all behave differently. Energy usage patterns will also vary by region, and as tech evolves."
4,solar,high,energy,0.962427,"Utilities already require prediction of variable loads and a variety of power resources that can change with clouds and weather. Longterm predictions Utilities construction projects power contracts requires planning for conditions decades in the future. Power resource markets, dispatch and management can be weeks, day ahead, or real time changing conditions to avoid costly and 3ven dangerous mistakes. UC San Diego has best program for real world research development renewable and microgrids...."
5,solar,high,Futurology,0.947308,"And then there is the 20 year life span of solar panels, and who will pay for the massive recycling process."
6,solar,high,climate,0.915851,People arguing against solar leave out the fact that with petroleum you destroy your product to use it.
7,solar,high,Futurology,0.996870,"> What solutions? I think the other user is getting at the hidden parts of the overall EV manufacturing and use cycle. How so? It still takes a lot of energy and raw materials to make an EV. So that still means mining/resource extraction, as well as the carbon footprint of that mining activity and the manufacturing process itself. To reduce the total environmental footprint, EV's must (eventually) run on sustainably produced electricity. Sources would be things like solar, hydro, wind, geoth..."
8,solar,high,energy,0.995793,"It matters if solar is used talking about ''green'' side of it. It matters much less if it is about getting electricity cheaply and securing better energy dependence. And yes, today you will mostly rely on China, but a solar panel bought today will still work at good capacity in 20 years and by that time you can set up production at home too if you want. Or if in 20 years China decides due to some politics not to sell you new panels it can still work at decent capacity until it is 26-27 year..."
9,solar,high,Futurology,0.989672,"Solar isn't meant to be a cure-all for energy needs. This being said, the more we utilize solar, wind, etc, the less GHGs we emit, mitigating further damage to the planet's atmosphere and environment as a whole."
